# 02 · Predicting late delivery

**Target.** `is_late = 1` when an order is delivered on a later calendar date than the estimate shown at checkout. Delivered orders only.

**Leakage rule.** Features must be known at purchase time. Excluded: approval, carrier hand-over and delivery timestamps, order status, reviews. Seller and route history only count orders that had *already been delivered* when this order was placed (`tests/test_leakage.py` checks this).

**Split.** By purchase date: train → validation → test. The threshold is picked on validation and applied unchanged to test.

In [ ]:
import sqlite3, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
DB = ROOT / "data" / "olist.db"          # built by: python src/etl.py
con = sqlite3.connect(DB)
q = lambda sql: pd.read_sql_query(sql, con)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
import features, train
df = features.build(DB)
df.to_csv(ROOT / "data" / "features.csv", index=False)
print(f"{len(df):,} delivered orders, late rate {df.is_late.mean():.2%}")
df.head()

In [ ]:
tr, va, te, vs, ts = train.time_split(df, None, None)   # or pass explicit dates, e.g. "2018-03-01", "2018-05-01"
fig, ax = plt.subplots()
m = df.set_index("purchase_ts").is_late.resample("MS").mean() * 100
ax.plot(m.index, m.values, marker="o")
for x, lab in [(vs, "val"), (ts, "test")]:
    ax.axvline(x, color="crimson", ls="--"); ax.text(x, ax.get_ylim()[1] * 0.95, f" {lab}", color="crimson")
ax.set_title("Late rate by month and split boundaries (watch for drift)"); ax.set_ylabel("% late")
plt.tight_layout(); plt.show()
print(len(tr), len(va), len(te))

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve
prior = tr.is_late.mean()
rate_cols = train.add_history_rates([tr, va, te], prior)
num, cat = features.NUMERIC_FEATURES + rate_cols, features.CATEGORICAL_FEATURES
X = num + cat

lr = train.logistic_baseline(num, cat).fit(tr[X], tr.is_late)
p_lr_va, p_lr_te = lr.predict_proba(va[X])[:, 1], lr.predict_proba(te[X])[:, 1]

train.as_category([tr, va, te], cat)
gbm, gbm_name, _ = train.fit_gbm(tr[X], tr.is_late, va[X], va.is_late)   # fit on train only here
p_gb_va, p_gb_te = gbm.predict_proba(va[X])[:, 1], gbm.predict_proba(te[X])[:, 1]
print(gbm_name)

In [ ]:
TARGET_RECALL = 0.5
rows = {}
for name, pv, pt in [("logistic", p_lr_va, p_lr_te), ("gbm", p_gb_va, p_gb_te)]:
    thr = train.threshold_for_recall(va.is_late, pv, TARGET_RECALL)
    rows[name] = train.evaluate(te.is_late, pt, thr)
pd.DataFrame(rows).T[["base_rate", "pr_auc", "roc_auc", "threshold", "precision", "recall", "flagged_pct", "precision_at_top10pct"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for name, p in [("Logistic", p_lr_te), (gbm_name.split(" (")[0], p_gb_te)]:
    pr, rc, _ = precision_recall_curve(te.is_late, p)
    ax.plot(rc, pr, label=f"{name} AP={average_precision_score(te.is_late, p):.3f}")
ax.axhline(te.is_late.mean(), ls="--", c="grey", label="base rate")
ax.set(xlabel="recall", ylabel="precision", title="Test PR curve"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### Threshold trade-off for operations
How many orders must ops review (flagged %) to catch a given share of late orders?

In [ ]:
out = []
for r in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    thr = train.threshold_for_recall(va.is_late, p_gb_va, r)
    e = train.evaluate(te.is_late, p_gb_te, thr)
    out.append({"target_recall(val)": r, "threshold": e["threshold"], "test_recall": e["recall"],
                "test_precision": e["precision"], "flagged_pct": e["flagged_pct"]})
pd.DataFrame(out)

In [ ]:
from sklearn.inspection import permutation_importance
s = va.sample(min(5000, len(va)), random_state=42)
imp = permutation_importance(gbm, s[X], s.is_late, scoring="average_precision", n_repeats=3, random_state=42)
fi = pd.Series(imp.importances_mean, index=X).sort_values()
fi.tail(15).plot.barh(figsize=(6, 5), title="Permutation importance (drop in PR-AUC)"); plt.tight_layout(); plt.show()

## Rolling backtest
One test window can mislead under drift. `python src/backtest.py` repeats the whole protocol on four consecutive two-month windows.

In [ ]:
bt = pd.read_csv(ROOT / "reports" / "backtest.csv")
display(bt[["test_window", "late_rate", "lr_pr_auc", "gbm_pr_auc", "gbm_lift", "gbm_roc_auc", "avg_promised_days"]])

## Limitations to write up
- **Right-censoring at the end of the data:** orders still in transit at the snapshot are excluded, and those skew late, so the last weeks of the test period look more on-time than reality.
- **Drift:** the late rate moves month to month (12% in Nov 2017, 19% in Mar 2018, 1% in Jun 2018) and Olist shortened its promises sharply in mid-2018, which reverses the sign of several geography features. A random split hides all of this (ROC-AUC 0.81 vs ~0.65–0.74 on time windows).
- **Estimate as a feature:** `promised_days` is legitimately known at checkout, but the model partly learns Olist's own estimation errors.
- **Geography** is zip-prefix centroids, not addresses.